# EEG Intersubject Mean-Variance Synchrony Analysis

This notebook implements the full **intersubject mean-variance synchrony** pipeline, covering:

* **Part 1 — Broadband (raw z-scored):** time-series overview, variance distribution,
  and windowed synchrony detection.
* **Part 2 — Per-frequency-band:** band-specific time series, variance distributions,
  pairwise ISC matrices, and windowed synchrony per band.

All computation uses `src.analysis.mean_variance` and all visualisation uses
`src.visualization.mean_variance_plots`.  The notebook is organised modularly: one
logical step per code cell so each cell can be run independently after the setup.

> **Parameters to tweak:** `CONDITION`, `MUSIC_TYPES`, `WINDOW_SEC`, `SYNC_PERCENTILE` in the
> *Configuration* cell below.

In [ ]:
import sys
import os

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from src.definitions.fields import (
    ConditionVariants,
    MusicTypeVariants,
    ExclusionCategories,
)
from src.analysis.mean_variance import (
    FREQUENCY_BANDS,
    compute_intersubject_stats,
    compute_windowed_stats,
    compute_band_intersubject_stats,
    compute_pairwise_isc_matrices,
)
from src.visualization.mean_variance_plots import (
    plot_timeseries,
    plot_variance_distribution,
    plot_windowed_analysis,
    plot_band_timeseries,
    plot_band_variance_distributions,
    plot_isc_matrices,
    plot_band_windowed_analysis,
)
from scripts.analysis_common import load_analyzers, analyzers_to_datasets

%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
print("Setup complete.")

## Configuration

In [ ]:
# ── Experiment configuration ─────────────────────────────────────────────────
CONDITION         = ConditionVariants.PLACEBO
MUSIC_TYPES       = [MusicTypeVariants.CLASSICAL, MusicTypeVariants.PSYTRANCE]
EXCLUSION_CATEGORIES = [ExclusionCategories.BAD_MUSIC, ExclusionCategories.ARTIFACTS]

# ── Windowed synchrony parameters ────────────────────────────────────────────
WINDOW_SEC        = 2.0    # non-overlapping window length in seconds
SYNC_PERCENTILE   = 10.0   # windows with variance < this percentile are "sync candidates"

# ── Data processing flag ──────────────────────────────────────────────────────
# Set True to load raw EDF files, resample, stack, and save before analysis.
# Keep False to use already-saved concatenated arrays.
process_and_save_data = False

## Data Loading

In [ ]:
# Load (or process-and-save) one EEGSummarizedAnalyzer per music type,
# normalise the data to z-scores (axis=2), and convert to AnalysisData.
analyzers = load_analyzers(
    MUSIC_TYPES,
    CONDITION,
    EXCLUSION_CATEGORIES,
    process_and_save_data,
    normalize_data=True,
)
datasets = analyzers_to_datasets(analyzers)
print("Available datasets:", list(datasets.keys()))

## Dataset Selection

Change `LABEL` to switch between music types.  The remaining cells use `ad` and the
derived variables `n_subjects`, `n_channels`, `n_times`.

In [ ]:
LABEL = MusicTypeVariants.CLASSICAL.value
# LABEL = MusicTypeVariants.PSYTRANCE.value

ad = datasets[LABEL]
n_subjects, n_channels, n_times = ad.data.shape
print(f"Dataset : {LABEL}")
print(f"Shape   : {ad.data.shape}  (subjects × channels × time points)")
print(f"Duration: {n_times / ad.sfreq:.1f} s  @  {ad.sfreq} Hz")

---
## Part 1 — Broadband (z-scored raw signal)

For the full broadband signal we compute:
* **Intersubject variance** at each `(channel, time)` cell — low values indicate synchrony
* **Per-channel-averaged time series** of mean and variance
* **Windowed statistics** with synchrony candidate labelling

### 1.1  Compute intersubject statistics

`compute_intersubject_stats` returns a dict with `inter_var`, `inter_mean`, `mean_t`,
`var_t`, `std_t`, and `mean_over_ch`.

In [ ]:
stats = compute_intersubject_stats(ad.data)

print(f"inter_var shape : {stats['inter_var'].shape}  (channels × time)")
print(f"mean_t    shape : {stats['mean_t'].shape}   (time)")
print(f"var_t     mean  : {stats['var_t'].mean():.4f}")
print(f"std_t     mean  : {stats['std_t'].mean():.4f}")

### 1.2  Time-series overview

Two-panel figure:
* **Top** — per-subject channel-average traces (light blue) + group mean (black) with ±1 SD band
* **Bottom** — channel-averaged intersubject variance over time

In [ ]:
fig_ts = plot_timeseries(stats, ad.sfreq, label=LABEL)

### 1.3  Intersubject variance distribution

Histogram of all `(channel × time)` variance values, clipped at the 99th percentile.

In [ ]:
fig_dist = plot_variance_distribution(stats["inter_var"], label=LABEL)

### 1.4  Windowed synchrony analysis

The recording is split into non-overlapping windows of `WINDOW_SEC` seconds.
A window is flagged as a **synchrony candidate** when its mean intersubject variance
falls below the `SYNC_PERCENTILE`-th percentile of the full `var_t` distribution.

In [ ]:
df_windows = compute_windowed_stats(
    stats,
    n_times=n_times,
    sfreq=ad.sfreq,
    window_sec=WINDOW_SEC,
    sync_percentile=SYNC_PERCENTILE,
)
n_sync = df_windows["sync_candidate"].sum()
print(f"Total windows   : {len(df_windows)}")
print(f"Sync candidates : {n_sync}  ({100 * n_sync / len(df_windows):.1f} %)")
df_windows.head(10)

#### Bar charts — per-window mean signal and mean variance

Green bars = synchrony candidates (mean variance below threshold).

In [ ]:
fig_bar, fig_overlay = plot_windowed_analysis(
    stats,
    df_windows,
    ad.sfreq,
    label=LABEL,
    window_sec=WINDOW_SEC,
    sync_percentile=SYNC_PERCENTILE,
)

---
## Part 2 — Per-Frequency-Band Analysis

The same analyses are repeated independently for each frequency band after
bandpass-filtering the z-scored data:

| Band  | Range |
|-------|-------|
| Delta | 1–4 Hz |
| Theta | 4–8 Hz |
| Alpha | 8–13 Hz |
| Beta  | 13–30 Hz |
| Gamma | 30–70 Hz |

### 2.1  Compute per-band intersubject statistics

`compute_band_intersubject_stats` applies `compute_intersubject_stats` to the
bandpass-filtered copy of `ad` for every entry in `FREQUENCY_BANDS`.

In [ ]:
band_stats = compute_band_intersubject_stats(ad)

print(f"{'Band':8s}  {'mean var_t':>12s}  {'mean mean_t':>12s}")
print("-" * 36)
for band, st in band_stats.items():
    print(f"{band:8s}  {st['var_t'].mean():12.4f}  {st['mean_t'].mean():12.4f}")

### 2.2  Per-band time-series overview

Five-row two-column figure.
* **Left column** — group-mean signal with per-subject traces and ±1 SD per band
* **Right column** — channel-averaged intersubject variance with synchrony threshold

In [ ]:
fig_band_ts = plot_band_timeseries(
    band_stats,
    ad.sfreq,
    label=LABEL,
    sync_percentile=SYNC_PERCENTILE,
)

### 2.3  Per-band intersubject variance distributions

One histogram per band in a 2-column grid, each clipped at the 99th percentile.

In [ ]:
fig_band_dist = plot_band_variance_distributions(band_stats, label=LABEL)

### 2.4  Pairwise inter-subject correlation (ISC) matrices

For z-scored data the mean product across channels and time equals the mean Pearson
correlation.  `compute_pairwise_isc_matrices` computes this for every subject pair
and frequency band.

In [ ]:
# Build {band: filtered_data_array} for the ISC computation
band_data_arrays = {
    band: ad.filter_to_band(l_freq, h_freq).data
    for band, (l_freq, h_freq) in FREQUENCY_BANDS.items()
}
isc_matrices = compute_pairwise_isc_matrices(band_data_arrays)

mask = ~np.eye(n_subjects, dtype=bool)
print(f"{'Band':8s}  {'off-diag mean ISC':>18s}")
print("-" * 30)
for band, mat in isc_matrices.items():
    print(f"{band:8s}  {mat[mask].mean():18.4f}")

In [ ]:
fig_isc = plot_isc_matrices(isc_matrices, n_subjects=n_subjects, label=LABEL)

### 2.5  Per-band windowed synchrony analysis

Two outputs:
1. **Summary figure** — all bands in one multi-row plot with windowed variance and
   synchrony candidate spans highlighted in green
2. **Per-band detailed figures** — one per band with continuous variance, windowed
   step function, sync threshold, and per-electrode variance heatmap

In [ ]:
fig_band_win_summary, per_band_figs = plot_band_windowed_analysis(
    band_stats,
    ad.sfreq,
    label=LABEL,
    window_sec=WINDOW_SEC,
    sync_percentile=SYNC_PERCENTILE,
)